<a href="https://colab.research.google.com/github/2xsec/2xsec.github.io/blob/master/02_Knowledge_Distillation_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Knowledge Distillation 실습**

- Response-Based Distillation은 큰 모델(Teacher)의 출력(response) 을 활용해 작은 모델(Student)을 학습시키는 지식 증류(Knowledge Distillation) 방법입니다.
- Knowledge Distillation(지식 증류) 기법을 사용하여, 성능이 좋은 **Teacher 모델(MobileNetV2)** 의 예측 정보를 더 작은 **Student 모델(LeNet)** 에 전달하는 과정을 실습합니다.



# 0. Colab 환경설정
- colab에서 GPU를 사용할 수 있도록 세팅
    - 런타임 > 런타임 유형 변경 > Python 3 와 T4 GPU 선택
- colab에서 Google Drive에 접근할 수 있도록 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("[현재 파일 위치]")
!pwd
print("[현재 디렉토리의 파일 확인]")
!ls

Mounted at /content/drive
[현재 파일 위치]
/content
[현재 디렉토리의 파일 확인]
drive  sample_data


**day 3** 폴더로 이동해주세요.

왼쪽의 **폴더** 아이콘을 클릭하면 경로를 쉽게 볼 수 있습니다.

In [ ]:
# 본인 환경에 맞게 경로를 수정하여 사용하세요.
%cd /content/drive/MyDrive/day3
!pwd

/content/drive/.shortcut-targets-by-id/17e7Sw1lHGD1Tob6tZfVVFZuLAUJ2zrAl/day3
/content/drive/.shortcut-targets-by-id/17e7Sw1lHGD1Tob6tZfVVFZuLAUJ2zrAl/day3


## 0-1. Setup
- 필요한 package를 import 합니다.
- Knowledge Distillation 실습에 필요한 주요 라이브러리를 불러옵니다.
- 이후 Teacher/Student 모델 정의, 데이터 로딩, 학습 및 평가를 위한 기본 환경을 구성하는 단계입니다.

In [ ]:
# 필요한 라이브러리들을 임포트합니다.
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn.functional as F
import os

In [ ]:
# 실험 재현성을 위한 랜덤 시드 고정
import random
import numpy

# 랜덤 시드를 고정합니다.
seed = 2026
random.seed(seed)
numpy.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 1. Teacher model - MobileNetV2

## 1-1. MobileNetV2의 기본 블록인 Conv-BN-ReLU 정의
- MobileNetV2를 구성할 때 반복적으로 사용되는 기본 합성곱 블록 `ConvBNReLU`를 정의합니다.
- 이 블록은 이후 MobileNetV2 내부에서 초기 feature extraction 및 depthwise separable convolution 구성에 활용됩니다.

이 블록은 다음 3개의 연산으로 구성됩니다.
1. `Conv2d`: 특징 추출을 위한 합성곱 연산
2. `BatchNorm2d`: 학습 안정화 및 정규화
3. `ReLU6`: 비선형 활성화 함수

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_planes, out_planes, kernel_size=3, stride=1, groups=1):
        ###### [실습] padding 계산: kernel_size의 크기에 따라 양쪽에 동일한 크기로 패딩을 설정합니다. ######
        padding = (kernel_size - 1) // 2
        ###### [실습] __init__ 메서드의 입력을 참고하여 Conv-BN-ReLU 구조를 완성하세요. ######
        super(ConvBNReLU, self).__init__(
            nn.Conv2d(
                in_planes,        # 입력 채널 수
                out_planes,       # 출력 채널 수
                kernel_size,        # 커널 크기
                stride,             # 스트라이드
                padding,            # 패딩 크기
                groups=groups,             # 그룹 설정
                bias=False,           # 바이어스는 사용X
            ),
            nn.BatchNorm2d(out_planes),  # Batch Normalization 레이어
            nn.ReLU6(inplace=True),      # 활성화 함수: ReLU6
        )

## 1-2. MobileNetV2의 핵심 블록인 Inverted Residual 정의
- MobileNetV2의 핵심 구조인 `InvertedResidual` 블록을 정의합니다.


- MobileNetV2의 Inverted Residual은 다음과 같은 특징을 가집니다.

1. 입력 채널을 먼저 확장(expansion)
2. depthwise convolution으로 연산량 절감
3. pointwise convolution으로 출력 채널 수 조정
4. 조건이 맞으면 residual connection 적용


In [ ]:
class InvertedResidual(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio):
        # Inverted Residual 블록의 초기화를 수행합니다.
        super(InvertedResidual, self).__init__()
        self.stride = stride
        assert stride in [1, 2]  ###### [실습] stride는 1 또는 2만 허용합니다. ######

        ###### [실습] 확장 비율(expand_ratio)을 적용해 hidden_dim을 설정하세요. ######
        hidden_dim = int(round(inp * expand_ratio))
        # ResNet처럼 입력과 출력을 연결할지 여부를 결정합니다.
        self.use_res_connect = self.stride == 1 and inp == oup

        # 레이어 설정 리스트를 초기화합니다.
        layers = []
        if expand_ratio != 1:
            layers.append(ConvBNReLU(inp, hidden_dim, kernel_size=1))
        layers.extend(
            [
                ConvBNReLU(hidden_dim, hidden_dim, stride=stride, groups=hidden_dim),
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
            ]
        )

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        # Residual 연결을 사용할 경우, 입력에 출력을 더하여 반환합니다.
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)

## 1-3. MobileNetV2 정의
Teacher 모델로 사용할 **MobileNetV2** 구조를 정의합니다.



In [ ]:
class MobileNetV2(nn.Module):
    def __init__(self, num_classes=10, width_mult=1.0):
        # MobileNetV2 네트워크 초기화를 수행합니다.
        super(MobileNetV2, self).__init__()
        block = InvertedResidual
        input_channel = 32
        last_channel = 1280

        # CIFAR10에 적합한 구조로 설정합니다.
        inverted_residual_setting = [
            # t (확장 비율), c (출력 채널 수), n (반복 횟수), s (스트라이드)
            [1, 16, 1, 1],
            [6, 24, 2, 1],  # CIFAR-10에 맞추어 Stride 2를 1로 변경합니다.
            [6, 32, 3, 2],
            [6, 64, 4, 2],
            [6, 96, 3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]

        # 첫 번째 레이어를 설정합니다.
        input_channel = int(input_channel * width_mult)
        self.last_channel = int(last_channel * max(1.0, width_mult))

        ###### [실습] CIFAR10용 첫 레이어는 stride를 1로 사용합니다. ######
        features = [ConvBNReLU(3, input_channel, stride=1)]

        # Inverted Residual 블록을 추가합니다.
        for t, c, n, s in inverted_residual_setting:
            output_channel = int(c * width_mult)
            for i in range(n):
                stride = s if i == 0 else 1
                features.append(
                    block(input_channel, output_channel, stride, expand_ratio=t)
                )
                input_channel = output_channel

        features.append(ConvBNReLU(input_channel, self.last_channel, kernel_size=1))
        self.features = nn.Sequential(*features)

        # classifier를 정의합니다.
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),                    # Dropout 레이어
            nn.Linear(self.last_channel, num_classes),  ###### [실습] 최종 출력층은 class의 개수(num_classes) ######
        )

        # 가중치를 초기화 합니다.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # 특징 추출 레이어를 거쳐 평균 풀링을 수행합니다.
        x = self.features(x)
        x = x.mean([2, 3])
        # 최종 분류기를 통해 클래스 예측값을 반환합니다.
        x = self.classifier(x)
        return x


## 1-4. Teacher Model 로드

In [ ]:
# Teacher model을 불러오고 파라미터를 고정합니다.
teacher_model = MobileNetV2()  ###### [실습] MobileNetV2 모델을 인스턴스화합니다. ######
teacher_model.load_state_dict(torch.load("./teacher_model.pt"))  # 저장된 모델 가중치를 로드합니다.
for param in teacher_model.parameters():   ###### [실습] teacher_model의 파라미터들을 불러옵니다. ######
    param.requires_grad = False  ###### [실습] False로 설정하여 학습 중 업데이트되지 않도록 고정합니다. ######
teacher_model = teacher_model.cuda().eval()  # 모델을 GPU로 옮기고, 평가 모드로 전환하여 Dropout 등의 레이어가 고정된 상태로 작동하게 합니다.


# 2. Student model - LeNet

## 2-1. Student 모델로 사용할 LeNet 정의
- Student 모델로 사용할 **LeNet** 구조를 정의합니다.  
- LeNet은 MobileNetV2보다 훨씬 단순하고 작은 CNN 구조입니다.

In [ ]:
# Student model을 정의합니다. (LeNet)
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        # Convolution 및 Pooling 레이어를 정의합니다.
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool2 = nn.MaxPool2d(2, 2)
        # Fully connected 레이어를 정의합니다.
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)  # FC 레이어를 위한 텐서 형태를 변환해줍니다.
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

## 2-2. Student 모델 생성
- 비교 실험을 위한 Student 모델 2개를 초기화 합니다.
- 두 모델을 따로 두는 이유는 **지식 증류를 적용한 경우와 적용하지 않은 경우를 공정하게 비교**하기 위해서입니다.

- `student_model_distill`: Knowledge Distillation을 적용하여 학습할 모델
- `student_model_alone`: Teacher 없이 일반 지도학습만으로 학습할 모델

In [ ]:
# Knowledge Distillation과 Standalone 학습을 위한 각각의 Student model을 초기화 합니다.
###### [실습] Student model은 LeNet 모델을 사용합니다. ######
student_model_distill = LeNet().cuda()
student_model_alone = LeNet().cuda()

# 3. 데이터셋 준비 및 관련 함수 정의

## 3-1. Knowledge Distillation 손실 함수 정의


In [ ]:
# Knowledge Distillation 손실 함수를 정의합니다.
class DistillationLoss(nn.Module):
    def __init__(self, temperature=3, alpha=0.7):
        super(DistillationLoss, self).__init__()
        self.temperature = temperature  # Temperature 값을 설정합니다.
        self.alpha = alpha  # Distillation loss와 Cross entropy loss 간의 가중치를 설정합니다.
        self.criterion = nn.CrossEntropyLoss()  ###### [실습] Cross entropy loss를 계산합니다. ######
        self.kd_loss = nn.KLDivLoss(reduction='batchmean')  ###### [실습] Distillation loss (KL Divergence)는 KLDivLoss로 계산합니다. ######

    def forward(self, student_outputs, teacher_outputs, labels):
        ###### [실습] softmax 함수를 활용하여 Teacher model의 출력을 소프트 라벨로 변환합니다. ######
        soft_labels = torch.softmax(teacher_outputs / self.temperature, dim=1)
        # Student logit을 계산합니다.
        ###### [실습] Student model 출력을 temperation 값으로 나누세요. ######
        student_logits = torch.log_softmax(student_outputs / self.temperation, dim=1)
        # Distillation loss를 계산합니다.
        distillation_loss = self.kd_loss(student_logits, soft_labels) * (self.temperature ** 2)
        # Student model의 학습 손실을 계산합니다.
        student_loss = self.criterion(student_outputs, labels)
        # 최종 손실을 반환합니다.
        return self.alpha * distillation_loss + (1 - self.alpha) * student_loss

## 3-2. CIFAR-10 데이터셋 준비

In [ ]:
# CIFAR-10 데이터셋을 불러옵니다.
transform = transforms.Compose([transforms.ToTensor()])  # 이미지 데이터를 텐서로 변환하는 전처리 과정을 거쳐줍니다.
train_dataset = CIFAR10(root='/content/data', train=True, download=True, transform=transform)
valid_dataset = CIFAR10(root='/content/data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, drop_last=True, num_workers=2)

100%|██████████| 170M/170M [1:35:15<00:00, 29.8kB/s]


## 3-3. 옵티마이저와 손실 함수 정의

In [ ]:
# 옵티마이저와 손실 함수를 정의합니다.
optimizer_distill = optim.Adam(student_model_distill.parameters(), lr=0.001)  # Distillation 모델용 옵티마이저
optimizer_alone = optim.Adam(student_model_alone.parameters(), lr=0.001)      # Standalone 모델용 옵티마이저
distillation_loss_fn = DistillationLoss(temperature=1, alpha=0.3)  # Distillation loss
standard_loss_fn = nn.CrossEntropyLoss()  # Cross entropy loss

## 3-4.  epoch 동안 모델을 학습하는 함수 정의
- 모델을 한 epoch 동안 학습시키는 `train_epoch` 함수를 정의합니다.
- 이 함수는 Teacher 모델을 사용할 경우와 사용하지 않을 경우를 모두 처리할 수 있도록 설계되었습니다.

In [ ]:
# 학습 함수를 정의합니다.
def train_epoch(model, train_loader, optimizer, criterion, teacher_model=None):
    model.train()  ###### [실습] 모델을 학습 모드(train)로 설정하세요. ######
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.cuda(), labels.cuda()  # 이미지와 라벨을 GPU로 이동합니다.

        if teacher_model:
            # Knowledge Distillation: Teacher model의 예측값을 사용합니다.
            with torch.no_grad():
                teacher_outputs = teacher_model(images)  ###### [실습] Teacher model의 출력 ######
            student_outputs = model(images)  # Student model의 출력
            ####### [실습] student_outputs와 teacher_outputs로 Distillation loss를 계산하세요. ######
            loss = criterion(student_outputs, teacher_outputs, labels)
        else:
            # Standalone을 학습합니다.
            student_outputs = model(images)
            ###### [실습] student_outputs와 labels로 loss를 계산하세요. ######
            loss = criterion(student_outputs, labels)

        optimizer.zero_grad()  # 그래디언트를 초기화합니다.
        loss.backward()        # 역전파를 수행합니다.
        optimizer.step()       # 파라미터를 업데이트합니다.

        total_loss += loss.item()
        _, predicted = torch.max(student_outputs, 1)  # 예측값을 계산합니다.
        total += labels.size(0)
        correct += (predicted == labels).sum().item()  ###### [실습] 정확도 계산을 위해 올바른 예측 수를 합산(sum)하세요. ######

    avg_loss = total_loss / len(train_loader)  # 평균 손실을 계산합니다.
    accuracy = 100 * correct / total           # 정확도를 계산합니다.
    return avg_loss, accuracy


## 3-5. 평가 함수 정의

In [ ]:
# 평가 함수를 정의합니다.
def evaluate(model, valid_loader):
    model.eval()  ###### [실습] 모델을 평가 모드(eval)로 설정합니다. ######
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.cuda(), labels.cuda()
            ###### [실습] 모델에 입력 데이터를 전달하세요. ######
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total  # 정확도를 계산합니다.
    return accuracy

# 4. Distillation 학습과 Student 단독 학습 수행 및 성능 비교

In [ ]:
EPOCHS = 5
# Knowledge Distillation을 사용한 학습을 진행합니다.
print("Training with knowledge distillation...")
for epoch in range(EPOCHS):
    distill_loss, distill_accuracy = train_epoch(student_model_distill, train_loader, optimizer_distill, distillation_loss_fn, teacher_model=teacher_model)
    print(f"Epoch {epoch+1}, Distillation Loss: {distill_loss:.4f}, Distillation Accuracy: {distill_accuracy:.2f}%")

# Distillation 모델의 최종 테스트 정확도를 평가합니다.
###### [실습] student_model_distill, valid_loader를 evaluate 함수에 넣어 최종 테스트 정확도를 계산합니다. ######
valid_accuracy_distill = evaluate(student_model_distill, valid_loader)
print(f"Final Distillation Model Test Accuracy: {valid_accuracy_distill:.2f}%")

# Knowledge Distillation 없이 Student model을 단독으로 학습합니다.
print("\nTraining student model alone...")
for epoch in range(EPOCHS):
    alone_loss, alone_accuracy = train_epoch(student_model_alone, train_loader, optimizer_alone, standard_loss_fn)
    print(f"Epoch {epoch+1}, Standalone Loss: {alone_loss:.4f}, Standalone Accuracy: {alone_accuracy:.2f}%")

# Standalone 모델의 최종 테스트 정확도를 평가합니다.
###### [실습] student_model_alone, valid_loader를 evaluate 함수에 넣어 최종 테스트 정확도를 계산합니다. ######
valid_accuracy_alone = evaluate(student_model_alone, valid_loader)
print(f"Final Standalone Model Test Accuracy: {valid_accuracy_alone:.2f}%")

Training with knowledge distillation...


Training:   0%|          | 0/781 [00:00<?, ?it/s]


AttributeError: 'DistillationLoss' object has no attribute 'temperation'